In [ ]:
!pip -q install -U \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-chroma \
    chromadb \
    pypdf \
    sentence-transformers \
    langchain-huggingface \
    langchain-google-genai

In [ ]:
from pathlib import Path

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
uploaded = files.upload()

Saving 2506.18027v3.pdf to 2506.18027v3 (3).pdf


In [ ]:
pdf_path = next(iter(uploaded.keys()))

print("Uploaded file:", pdf_path)

Uploaded file: 2506.18027v3 (3).pdf


In [ ]:
loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("Number of pages:", len(documents))

Number of pages: 14


In [ ]:
print(documents[0].page_content[:2000])

PDF Retrieval Augmented Question Answering
Thi Thu Uyen Hoang Meenakshi Rajendran Kun Zhang Yuhan Wu Viet Anh Nguyen
Saarland University
{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de
Abstract
This paper presents an advancement in Question-Answering (QA) systems using
a Retrieval Augmented Generation (RAG) framework to enhance information
extraction from PDF files. Recognizing the richness and diversity of data within
PDFs—including text, images, vector diagrams, graphs, and tables—poses unique
challenges for existing QA systems primarily designed for textual content. We seek
to develop a comprehensive RAG-based QA system that will effectively address
complex multi-modal questions, where several data types are put together in the
question. This is mainly achieved by refining approaches toward processing and
integrating non-textual elements in PDFs into the RAG framework to derive precise
and relevant answers, as well as finetuning large language models to 

In [ ]:
print(documents[0].metadata)

{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Thi Thu Uyen Hoang; Meenakshi Rajendran; Kun Zhang; Yuhan Wu; Viet Anh Nguyen', 'doi': 'https://doi.org/10.48550/arXiv.2506.18027', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'PDF Retrieval Augmented Question Answering', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2506.18027v3', 'source': '2506.18027v3 (3).pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}


In [ ]:
for i, doc in enumerate(documents[:3]):
    print("=" * 60)
    print("PAGE:", i + 1)
    print("=" * 60)
    print(doc.page_content[:1000])

PAGE: 1
PDF Retrieval Augmented Question Answering
Thi Thu Uyen Hoang Meenakshi Rajendran Kun Zhang Yuhan Wu Viet Anh Nguyen
Saarland University
{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de
Abstract
This paper presents an advancement in Question-Answering (QA) systems using
a Retrieval Augmented Generation (RAG) framework to enhance information
extraction from PDF files. Recognizing the richness and diversity of data within
PDFs—including text, images, vector diagrams, graphs, and tables—poses unique
challenges for existing QA systems primarily designed for textual content. We seek
to develop a comprehensive RAG-based QA system that will effectively address
complex multi-modal questions, where several data types are put together in the
question. This is mainly achieved by refining approaches toward processing and
integrating non-textual elements in PDFs into the RAG framework to derive precise
and relevant answers, as well as finetuning large language mo

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Pages:", len(documents))
print("Chunks:", len(chunks))

Pages: 14
Chunks: 67


In [ ]:
# from langchain_text_splitters import CharacterTextSplitter

# splitter = CharacterTextSplitter(
#     chunk_size=800,
#     chunk_overlap=100
# )

# chunks = splitter.split_documents(documents)

# print("Pages:", len(documents))
# print("Chunks:", len(chunks))

In [ ]:
# from langchain_text_splitters import NLTKTextSplitter
# import nltk
# nltk.download('punkt_tab')

# splitter = NLTKTextSplitter(
#     chunk_size=800,
#     chunk_overlap=100
# )

# chunks = splitter.split_documents(documents)

# print("Pages:", len(documents))
# print("Chunks:", len(chunks))

In [ ]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 60)
    print("CHUNK:", i + 1)
    print("=" * 60)
    print(chunk.page_content[:1000])

CHUNK: 1
PDF Retrieval Augmented Question Answering
Thi Thu Uyen Hoang Meenakshi Rajendran Kun Zhang Yuhan Wu Viet Anh Nguyen
Saarland University
{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de
Abstract
This paper presents an advancement in Question-Answering (QA) systems using
a Retrieval Augmented Generation (RAG) framework to enhance information
extraction from PDF files. Recognizing the richness and diversity of data within
PDFs—including text, images, vector diagrams, graphs, and tables—poses unique
challenges for existing QA systems primarily designed for textual content. We seek
to develop a comprehensive RAG-based QA system that will effectively address
complex multi-modal questions, where several data types are put together in the
CHUNK: 2
complex multi-modal questions, where several data types are put together in the
question. This is mainly achieved by refining approaches toward processing and
integrating non-textual elements in PDFs into the RAG

In [ ]:
for size in [300, 800, 1500]:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=100
    )

    test_chunks = splitter.split_documents(documents)

    print(
        f"Chunk size = {size} "
        f"| Number of chunks = {len(test_chunks)}"
    )

Chunk size = 300 | Number of chunks = 220
Chunk size = 800 | Number of chunks = 67
Chunk size = 1500 | Number of chunks = 37


In [ ]:
embedding_model_name = "BAAI/bge-m3"

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

print("BGE-M3 embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not available. Using CPU.")

CUDA available: False


In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

print("BGE-M3 loaded successfully.")

Using device: cpu


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE-M3 loaded successfully.


In [ ]:
text = "What is the attendance requirement?"

vector = embeddings.embed_query(text)



print("Vector type:", type(vector))
print("Vector dimensions:", len(vector))
print("First 10 values:", vector[:10])

Vector type: <class 'list'>
Vector dimensions: 1024
First 10 values: [-0.0008787871920503676, -0.016933498904109, -0.049234889447689056, -0.002397366100922227, -0.0075664944015443325, -0.03898830711841583, -0.004098901059478521, -0.03445146605372429, 0.02240793965756893, 0.010130854323506355]


In [ ]:
text1 = "What is the attendance requirement?"
text2 = "How much attendance do students need?"
text3 = "What time does the library open?"

v1 = embeddings.embed_query(text1)
v2 = embeddings.embed_query(text2)
v3 = embeddings.embed_query(text3)

print("Vector 1:", len(v1))
print("Vector 2:", len(v2))
print("Vector 3:", len(v3))

Vector 1: 1024
Vector 2: 1024
Vector 3: 1024


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_12 = cosine_similarity([v1], [v2])[0][0]*100
similarity_13 = cosine_similarity([v1], [v3])[0][0]*100

print("Question 1 vs Question 2:", similarity_12)
print("Question 1 vs Question 3:", similarity_13)

Question 1 vs Question 2: 84.76967865855276
Question 1 vs Question 3: 53.895783812049125


In [ ]:
import shutil
import os

if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

print("Old ChromaDB deleted.")

persist_directory = "./tmp12/chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

print("Vector store created successfully.")

Old ChromaDB deleted.
Vector store created successfully.


In [ ]:
question = "What is Recent progress in machine learning and natural language processing  ?"

results = vectorstore.similarity_search(
    question,
    k=3
)

In [ ]:
print(documents[0].page_content[:2000])

PDF Retrieval Augmented Question Answering
Thi Thu Uyen Hoang Meenakshi Rajendran Kun Zhang Yuhan Wu Viet Anh Nguyen
Saarland University
{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de
Abstract
This paper presents an advancement in Question-Answering (QA) systems using
a Retrieval Augmented Generation (RAG) framework to enhance information
extraction from PDF files. Recognizing the richness and diversity of data within
PDFs—including text, images, vector diagrams, graphs, and tables—poses unique
challenges for existing QA systems primarily designed for textual content. We seek
to develop a comprehensive RAG-based QA system that will effectively address
complex multi-modal questions, where several data types are put together in the
question. This is mainly achieved by refining approaches toward processing and
integrating non-textual elements in PDFs into the RAG framework to derive precise
and relevant answers, as well as finetuning large language models to 

In [ ]:
for i, result in enumerate(results):
    print("=" * 60)
    print("RESULT:", i + 1)
    print("=" * 60)
    print(result.page_content)
    print("\nMetadata:", result.metadata)

RESULT: 1
in multimodal data integration and processing.
1 Introduction
Recent progress in machine learning and natural language processing has remarkably improved
interactions with digital documents leading to better information retrieval systems. The most
important aspect is the Retrieval Augmented Generation (RAG) framework Lewis et al. [2020]
for QA systems, which combines both retrieval and generation-based approaches for handling
difficult questions. In our work, we enhance the existing RAG-based QA system for information
extraction through text, images, vector diagrams/graphs, and tables provided in PDFs.
RAG is designed to address the serious limitations of the large language models (LLMs) such as

Metadata: {'author': 'Thi Thu Uyen Hoang; Meenakshi Rajendran; Kun Zhang; Yuhan Wu; Viet Anh Nguyen', 'page': 0, 'arxivid': 'https://arxiv.org/abs/2506.18027v3', 'doi': 'https://doi.org/10.48550/arXiv.2506.18027', 'total_pages': 14, 'trapped': '/False', 'creationdate': '', 'creator':

In [ ]:
question = "What is the Forecasting and Data Modeling?"

for k in [1, 3, 5]:

    results = vectorstore.similarity_search(
        question,
        k=k
    )

    print(f"\n===== k = {k} =====")

    for i, result in enumerate(results):
        print(f"\nResult {i + 1}:")
        print(result.page_content[:300])


===== k = 1 =====

Result 1:
is to develop a comprehensive system capable of answering complex, multifaceted questions that
necessitate the integration, retrieval and interpretation of diverse data types.
2

===== k = 3 =====

Result 1:
is to develop a comprehensive system capable of answering complex, multifaceted questions that
necessitate the integration, retrieval and interpretation of diverse data types.
2

Result 2:
is to develop a comprehensive system capable of answering complex, multifaceted questions that
necessitate the integration, retrieval and interpretation of diverse data types.
2

Result 3:
is to develop a comprehensive system capable of answering complex, multifaceted questions that
necessitate the integration, retrieval and interpretation of diverse data types.
2

===== k = 5 =====

Result 1:
is to develop a comprehensive system capable of answering complex, multifaceted questions that
necessitate the integration, retrieval and interpretation of diverse data types.


In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass(
    "Enter your Google API key: "
)

Enter your Google API key: ··········


In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

In [ ]:
response = llm.invoke(
    "Explain RAG in two simple sentences."
)

print(response.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Retrieval-Augmented Generation (RAG) is a technique that lets an AI look up relevant facts from an external database before answering a prompt. This ensures the AI provides accurate, up-to-date answers based on real information rather than relying only on its memory.', 'extras': {'signature': 'EtAVCs0VARFNMg8+3CpcM8KDyxxfHwoFQ9fLyVuVNSiYcvubjHUnHGcz90HbaBCRAiZjLPl3+BTaNy/Od8Eu+uaZelfGn5gcJTbpDgbuRnyOQ/Cxfoc/wSEpOZWOfMjoR9ixyOSCn9hHRFW7vyyf7Zlp3XZpf0zCWISGkT1uzXx3JWWnYD3ZeU2wXta9jOKs7eW8ZJJ7jiwHbuMMfeZqChngVj2dqWJGq1ElrdhnuLWHtJMzxyjifpD69bPyOpQZox9gzOQ33jkdo237TDtoASOj9vWVRIo5BZVw1AktP4ZVh7D8GlyAHQTPWEwFBZqm6yL2hh1TG/jz2UsjQl5/7dYPwkDQhtHXqDngB4HtlmxyWbKGaYEDs5tWhn082HjmNjP+lPMMeVl4sytipWBSAWWk8V610MDvAd7A+5DhiOtkC8u6jM9bAnrV0JXK3HK8lkvSwngxtZmeJFN9Os77MsM6xuOF2Dp0sYjdRYBH7YKLHMiiG5KOAZDZ+8NfXPGKzOwRsbKbbCEkCbjpjyRljgW1CodDabg+RZp0VlxtGlJdsYOR4BDHqye9M5qUNEEz8xjaPUuPMenYWHDnoD2x3eiF90MqpUseEaKeWl0X6SNt8+AkGBG7YZdrO6iYn2TJ2c3I8qFuh7JXXENS2Slg57wAa7obBe3s9ku6y7

In [ ]:
question = "why is  removal of headers and footers is an important?"

results = vectorstore.similarity_search(
    question,
    k=3
)

context = "\n\n".join(
    document.page_content
    for document in results
)

print(context)

3.2.1 PDF Preprocessing
Figure 2: PDF Preprocessing
Header and footer removal
The removal of headers and footers is an important preprocessing step as they could interfere with the
retrieval process by adding noise to the data which leads to less accurate results. Thus, by removing
headers and footers, we obtain reliable cleaned pdf documents for further processing. We make
an assumption that headers and footers coordinates are consistent across pages, i.e., at the top and
bottom of the page, therefore by detecting this repeating pattern we will be able to remove headers
and footers. As shown in Figure 2, we employ DBSCAN (Density-Based Spatial Clustering of
Applications with Noise) which is well known for identifying areas of high density Fahim [2022]. We

3.2.1 PDF Preprocessing
Figure 2: PDF Preprocessing
Header and footer removal
The removal of headers and footers is an important preprocessing step as they could interfere with the
retrieval process by adding noise to the data which

In [ ]:
prompt = f"""
You are a helpful document assistant.

Answer the question using ONLY the context provided below.

If the answer is not present in the context,
say that you could not find the answer in the document.

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
print(prompt)


You are a helpful document assistant.

Answer the question using ONLY the context provided below.

If the answer is not present in the context,
say that you could not find the answer in the document.

Context:
3.2.1 PDF Preprocessing
Figure 2: PDF Preprocessing
Header and footer removal
The removal of headers and footers is an important preprocessing step as they could interfere with the
retrieval process by adding noise to the data which leads to less accurate results. Thus, by removing
headers and footers, we obtain reliable cleaned pdf documents for further processing. We make
an assumption that headers and footers coordinates are consistent across pages, i.e., at the top and
bottom of the page, therefore by detecting this repeating pattern we will be able to remove headers
and footers. As shown in Figure 2, we employ DBSCAN (Density-Based Spatial Clustering of
Applications with Noise) which is well known for identifying areas of high density Fahim [2022]. We

3.2.1 PDF Preprocessi

In [ ]:
response = llm.invoke(prompt)

print(response.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Based on the provided context, the removal of headers and footers is important because:\n\n* They could interfere with the retrieval process by adding noise to the data, which leads to less accurate results.\n* Removing them allows for obtaining reliable, cleaned PDF documents for further processing.', 'extras': {'signature': 'EtMLCtALARFNMg//5fRrP9+MB3vu0lyrg1gqLy/PZtEIUqWShMJ4Gp7sPghplulb00HteYfyI8/EYK8Yft6XQmd/QVQl9a/L7T9DlM5TBkGv3sZLV8E3Da8TKRaFOO7E2po7B+45TRhM0ESr1KcNk9a9JRdiAhu0Clf4JcQ29oiMWAwNGLq0DIbYv80iYnRIl+CS7uxN1FMpmwc2+fWG6OVTnQzcHPH0k4AKKPDKXnEpT+7xywciBZIeXL+tcf5/Yd5ZRNt3v0CAKeYDQWED6w1o+nZLeFjJh/cqGf9rGVWDSmFqkfTObDPAV7LEjImiuRjcqp9XBSkto1Gp3E0msXs8zijlUgLsCTaaJXMH3qfqhPpmXwVwkAUX/tlMw5myQYdMhCVMwo7ffClJNghoXlxjgqjDs24DHcFkcdkba1qhpnMGy7n3npesF+QtMyX2Q7O8QD62zBvY8VpK+18TM7/1bja8nXIuKFvOdR/5iFF0FIO23BZVrMr/xe7ikv5S/YVYVdpNQbUZftyENYJCGpQg+7NcMPG17wv0peUb2iQFPSUXZV9OkpJcqFpNtMc7jdYJnWqKK0Fll05rrywjBhxSIH7DHymfvRyl1gKcsM+j2pZPC5qbFl7GZJayKFdBvSl1

In [ ]:
def ask_rag(question, k=3):

    # Retrieve relevant chunks
    results = vectorstore.similarity_search(
        question,
        k=k
    )

    # Build context
    context = "\n\n".join(
        doc.page_content
        for doc in results
    )

    # Prompt
    prompt = f"""
You are a helpful document assistant.

Answer ONLY using the context below.

If the answer is not available in the context,
say you could not find it in the document.

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate
    response = llm.invoke(prompt)

    return response.content, results

In [ ]:
answer, sources = ask_rag(
    "why is  removal of headers and footers is an important??"
)

print(answer)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Based on the provided context, the removal of headers and footers is an important preprocessing step because they could interfere with the retrieval process by adding noise to the data, which leads to less accurate results. Removing them allows for obtaining reliable, cleaned PDF documents for further processing.', 'extras': {'signature': 'EqUPCqIPARFNMg/ggir6dLg+4SjETGNGe5iC5IHv3EZtTVQ9bmdSrOWQtR1ID5fRQZBWVwtM1KnG/eQ3kxDfYb67lyXSID1Ih6lY9J0R5In6VP0SpMsUcXpzaXnk99da7+qxs1HaLjhJQyVeEymoCoImXZ8qu252EHnH01f0CE+98evoTmeyq70Oj7JqxGdXF+YvmGHatbSG93WF19faRK9q9nNhehp9CENvnNG7Ig4CA3r0s3AgaXKi6nlk7LD2K0luWaxKmmV1amc31yTvjAtAIAfAUf2hX9ya3w0QgWHvTDZ4zAAzHBljbo/REw4oFk+m+EM0rDY6gXwq4JLyijkXo5hLrz5rkIgjzW2OgqXMxOT0/Kb1vui2cfw7KedZ1d+HZjxDJFvmB1e8azZcL1iYyhDVw1thrlzhd+MMTzBFdXdDV+bDNjjJy+Wnyy2/sMmEK9BQ7jiw5nAd5oXlVxiTd54WrkL8s43Kff1QET0Ob5pd/AUPpqG9CJUzrMjrZYJLiETsb2IKrjJKDCq5uu8zjj9ROy/zMl2mzZsKqCUTqMqcOXlpz4Em3bztcmzP+vfZOUlw4hgPml4U0kzdyNQZb/1Mrh23XHlMlvqzP7+BPxo3BKQbGr6

In [ ]:
answer, sources = ask_rag(
    "why is  removal of headers and footers is an important???"
)

print("ANSWER")
print("=" * 60)
print(answer)

print("\nSOURCES")
print("=" * 60)

for i, source in enumerate(sources):
    print(
        f"Source {i + 1}:",
        source.metadata
    )

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


ANSWER
[{'type': 'text', 'text': 'Based on the provided context, the removal of headers and footers is an important preprocessing step because headers and footers could interfere with the retrieval process by adding noise to the data, which leads to less accurate results. Removing them allows for obtaining reliable, cleaned PDF documents for further processing.', 'extras': {'signature': 'Et4LCtsLARFNMg+TfdEsliuSZsvE/aTbzZuAjbd2RqbR7nRV7kBo2O2XCBotnz7jSMKPwxc11bS31vBiqe/DwCM4x1dmZMPptykRb+fiFdINWQpfyqp1VlKb0NPTlywlKMHj5yaegAMZ8A805dXi4LC+DSsZ6yduhTMz6r+ec9zsCJx/JDOayYVj08wXSeu30bSKAyXbxkclv09yRCYeYsQyZHWFQpBRm2vSIZMUN82pd1/igFZA3wpllOh0UXP5Z/e5e34BvDwHxR6uH30wdhBZyWs0aFlymbM1/t9mhd5Y3OJZX+RgbwkSFkRCjbJxP4UZh86C9/+0wEx5gMHMgxe7nsTYYJ8ZfsrI7Hj4ocWkit3nWZq/Wg91jDFTD2m74ZbZPXLeFSjqc9GA3OyuGI2W6LkLb1Vg9g/EqhwslhoE8zXQ58epCg6nHFx0eDqxCRaiQVuuT8IfSGN3FYUvIshlSTJEvEJZldxaFh+GW+ax8DRJyUvi69DYXQp5tTbLdoBivW8HmnTpmwQkbMTONzeIKXI9XivmyQCC6B0V2BsA2VKdRrFd0ppl8k9uSLDtR9bV0XQ85YAj7s77denOm1r+RsJ3/q27v

In [ ]:
print("====================================")
print("       PDF RAG CHATBOT")
print("====================================")
print("Type 'exit' to stop.\n")

while True:

    question = input("You: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer, sources = ask_rag(
        question,
        k=3
    )

    print("\nAI:", answer)

    print("\nSources:")

    for source in sources:
        print(source.metadata)

    print()

       PDF RAG CHATBOT
Type 'exit' to stop.

You: "why is  removal of headers and footers is an important???" )


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



AI: [{'type': 'text', 'text': 'Based on the provided context, the removal of headers and footers is an important preprocessing step because they could interfere with the retrieval process by adding noise to the data, which leads to less accurate results. Removing them helps obtain reliable, cleaned PDF documents for further processing.', 'extras': {'signature': 'EpcMCpQMARFNMg+/t04tqIzkXuf1RLgB9RVwynWloaUsrbCNDixOy1blb4JxDhhpznxxaKB91KD2qy1VAga6ZOkXnGlBO6dYsYmxyb8ZcFVKocpubAmI/kbkks/VkfSJjVPNT0kDlI4vBgE6ob0evUjs29Eyj1xJy5Gg90Rv3N4xPb9fO6G8SzoeJkOEqtIoqguneH4WHfSOxPe0VqEgZzFCfFYqN3vSKV77H4Qmh9zA7SY4/89ahr9MYMkGl2bCOkkFtJbxGQTv95HOCwjPP1DHUS1Qc4RJzbX36na6IjwFiwQfNk8IySmo0jSdZ6Vi3h76NWq0yF9WAife9E61uhXDI+4GQ4tQoNZiXuxPDYiV22QTZkBZquHjUcf6HQREzWs3o7otcbety5ijgReDWu55VuBSIoM7faOgoHu8/4b2t04tILYS4NG0ROQDCzt+GsGwdL6LvWwRdMM0olR7BHQKsiz0tWJ5C9FYIW63rlYYQwkk5dBBT2nU80oBvdeBTCoyO3Wf2x1Ru2htWsvvuXLiQrU+NefbGPcr0HXBRBurl/azAeQ96j4DZXXlBaKAt5ouInR6e7G3gPyygmPKgIbb4I5q22aVu5g4cgEkCT2nen6rmdSjLtrsNi

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



AI: [{'type': 'text', 'text': 'I could not find it in the document.', 'extras': {'signature': 'EqkKCqYKARFNMg9hzgfVkkqRarW9+4WoL6Gmh90Pj2qwZkPxnQyeD9M/cKuKSJUSeP9YFZAwpndiSLFEWGzs5oKEfki4ls+6AwuAcO1LJHk/RIh3E94jDAwKOQCDkxnUXLsR3nZk0ZLOJOR3OSwyNCkKCH7TmIXmvoS/g+M8aOJfjS+/aXOLJzmQuFQ3IqpbWGtxqisw3BN67/3kGbedTG1KPfp8DLM4n4zqRvSb800JAi8DNBKzSBscOTJ/o81CjR/CsGcCzHqZqyxCubL6NOhq28fSeq9oy838AVK/IRhOyMTa/0kOp07yHyjvWcDcti5pA8h3MQmN3n1UKqIXlcGxsb5CckGBsU5/+z+2ChN3ZtyD9gocZz/KRjy/djLKQ6Grtgoeb9QU44Z3VVhSLTgaAE2KRyLDnFaudSY3BCwFeIcDmuIhi8UFrZfCqoxasGXCKNPGYje6qvn8jMih053A75/3hGRm67bLkBfpFsw3VJwuZpUbksKeohLAs1vdylzm5WU1bZ22f1W/0wmJmKCUwSBMdkHgYun8nYxzk/45qBxtr2uHHRQZMcrjmgZ2TBk7ylVeQhCfZKuyUvjbtRPLdsQb1ZPMCdiqK2JasMEveYKFMKdeToPkNkJP1oH588HYULOtbtseRzp2Ykyl73eWdq6YWrQc0DJgSAnvlV8eSPxJJPULbnaCQCDto2HIzzrNDqKcTa9Qq3CLhLF4dYe+LlqFvJpxnRTB1eqv6L4PRG0LazOG7c0bsuYPx6j26WLzguPg631Y0w329FmG748GXzrnpY6RmbdA1LVoBdiQYNE1ZCVGVTcyN7oLMCqqCdaAn0cPK/hw8VYxYRd9aUt4UZno1akq/mqHUjMOGcsmeXfM1wg27Xe+IkWBLEoTfW96BnVF

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



AI: [{'type': 'text', 'text': 'I could not find it in the document.', 'extras': {'signature': 'EvAHCu0HARFNMg+MKjmqQngMzl5dXeS+hBgZZCZAErK5HijlRBxq7zk0fn8+l8ASFdsYuHvRmNCxb6wzf+KgTlZjhPWKIVQraOCQOnsmWnPNbfYzEAEFVFE/Bhdraq4+ZhHAxk+qlfazUHmab6g14EJhSWeD/XctTNIlZY1DGxYAKvFQDlVptYv+yRM2Orwj/0wUS4i/9lieG/whnQ39z62GUEuJE9i4UwmRirClyKAcIPxStVqd+g4N4NQrqPoiUH2zA8cX/zgPEfrFd/BWVEEAq89JqRbEqVTD3CNL/YoV/82q95gfCDc+MpURqp4XadC0je/ugHGMyXOOsnv1b+CLdaRgLJTT3Afmqo9xsxQRDPzYTMlFGAEEUqr9y+ZaGxUNXUV/AxhWWo7rXc4FUlRr9tG6e39m52VKPGPq7jPXhxBLzGN9/AuSSZRlAKL7e2oVLSJSTbgk+ZnErdXoN7kDIyoehVNqCjLXcOekZySi/RY9tQpDZlr75Eqm+aje5Z29Uk3f3EA+osbKd4TeQ8JM+5UZm/knAvKzSszOAmXturAAi6TRWbd+YPFHn8ba7xgh4Jy40jgfGNKDJ2NbDJw4j3CfB8Du8w+0PCA7Hy0C/aBRDvp0w+s92qP5dL3Io1+fqAs+JnhUFkQWO+Vi9bOtP08S9M0p1drDQJXyRbZVwoAJjwXs86Pnxo9gyccgJDhJylEEUsyNu87zvBFIHuUGIFyuuyUwtT79NnkMKZQHkBYpLlH5GQ0bQ9zTEUJh1vyYiD3kWYAMOa5DZmIKWZYPifxFQE81R5LpK5/++NUND3qVebSlpGNXWcUIV3+2IQdIS+EElZU4j3WChpsbXGDxwitDnOdeuRV9WfwWmLmb51em+T231vqJxyc81UtyCvFuffhO

In [ ]:
\